In [0]:
#  Re-register all tables — run before any notebook 

STORAGE_ACCOUNT = "adlssupplychain"
CONTAINER       = "supplychain-data"
BASE_ABFSS      = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

spark.conf.set(
    "spark.databricks.delta.schema.autoMerge.enabled", "true")
spark.conf.set(
    "spark.databricks.delta.overwriteSchema.enabled",  "true")

# Re-create database with explicit ADLS location
spark.sql(f"""
    CREATE DATABASE IF NOT EXISTS supply_chain_db
    LOCATION '{BASE_ABFSS}/hive-warehouse/supply_chain_db.db'
""")

# All tables with their ADLS paths
tables = {
    "supply_chain_100gb":
        f"{BASE_ABFSS}/silver/supply_chain_100gb",
    "pagerank_results":
        f"{BASE_ABFSS}/gold/pagerank_results",
    "high_risk_paths":
        f"{BASE_ABFSS}/gold/high_risk_paths",
    "risk_clusters_transactional":
        f"{BASE_ABFSS}/gold/risk_clusters_transactional",
    "risk_clusters_graph_augmented":
        f"{BASE_ABFSS}/gold/risk_clusters_graph_augmented",
    "feature_catalog":
        f"{BASE_ABFSS}/gold/feature_catalog",
    "feature_importances":
        f"{BASE_ABFSS}/gold/feature_importances",
    "feature_stability":
        f"{BASE_ABFSS}/gold/feature_stability",
    "scalability_benchmarks":
        f"{BASE_ABFSS}/gold/scalability_benchmarks",
}

print("Re-registering tables...\n")
for table_name, path in tables.items():
    try:
        # Check Delta files exist
        dbutils.fs.ls(path + "/_delta_log")

        # Drop and recreate
        spark.sql(
            f"DROP TABLE IF EXISTS "
            f"supply_chain_db.{table_name}")
        spark.sql(f"""
            CREATE TABLE supply_chain_db.{table_name}
            USING DELTA
            LOCATION '{path}'
        """)
        count = spark.table(
            f"supply_chain_db.{table_name}").count()
        print(f"  OK  {table_name:45s} {count:>15,} rows")

    except Exception as e:
        print(f"  --  {table_name:45s} "
              f"(not found — will be created later)")

print("\nAll available tables registered.")
print("\nCurrent tables in supply_chain_db:")
spark.sql("SHOW TABLES IN supply_chain_db") \
     .show(truncate=False)

Re-registering tables...

  OK  supply_chain_100gb                                101,840,453 rows
  OK  pagerank_results                                          167 rows
  OK  high_risk_paths                                       338,663 rows
  OK  risk_clusters_transactional                               561 rows
  OK  risk_clusters_graph_augmented                             561 rows
  OK  feature_catalog                                            10 rows
  --  feature_importances                           (not found — will be created later)
  OK  feature_stability                                           6 rows
  OK  scalability_benchmarks                                      6 rows

All available tables registered.

Current tables in supply_chain_db:
+---------------+-----------------------------+-----------+
|database       |tableName                    |isTemporary|
+---------------+-----------------------------+-----------+
|supply_chain_db|feature_catalog              |false

In [0]:
# NOTEBOOK 05 — Genetic Algorithm Feature Discovery
# Phase A: Feature discovery on 1GB sample
# Phase B: Validation on 10GB
# Phase C: AUC stability check — proves features generalise
# This addresses the "100GB to find 5 features" criticism.


from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType
import mlflow, random, time, pandas as pd
import numpy as np

STORAGE_ACCOUNT     = "adlssupplychain"
CONTAINER           = "supplychain-data"
BASE_ABFSS          = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
GOLD_FEATURES_ABFSS = f"{BASE_ABFSS}/gold/feature_catalog"
GOLD_IMP_ABFSS      = f"{BASE_ABFSS}/gold/feature_importances"
GOLD_STABILITY_ABFSS= f"{BASE_ABFSS}/gold/feature_stability"

mlflow.set_experiment("/supply-chain-pipeline")

<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/1418298044073275', creation_time=1776371864602, experiment_id='1418298044073275', last_update_time=1776842087471, lifecycle_stage='active', name='/supply-chain-pipeline', tags={'mlflow.experiment.sourceName': '/supply-chain-pipeline',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'sudheeshudhayakumar07@gmail.com',
 'mlflow.ownerId': '145444312514419'}>

In [0]:
# SECTION 0 — Global config + clean slate


spark.conf.set(
    "spark.databricks.delta.schema.autoMerge.enabled", "true")
spark.conf.set(
    "spark.databricks.delta.overwriteSchema.enabled",  "true")

for tbl in [
    "supply_chain_db.feature_catalog",
    "supply_chain_db.feature_importances",
    "supply_chain_db.feature_stability",
]:
    spark.sql(f"DROP TABLE IF EXISTS {tbl}")

for path in [GOLD_FEATURES_ABFSS, GOLD_IMP_ABFSS,
             GOLD_STABILITY_ABFSS]:
    try:
        dbutils.fs.rm(path, recurse=True)
        print(f"  Cleared: {path.split('/')[-1]}")
    except Exception:
        print(f"  Already empty: {path.split('/')[-1]}")

spark.sql("CLEAR CACHE")
spark.catalog.clearCache()
print("Clean slate confirmed.\n")

  Cleared: feature_catalog
  Cleared: feature_importances
  Cleared: feature_stability
Clean slate confirmed.



In [0]:
# SECTION 1 — Auto-discover numeric columns


df = spark.table("supply_chain_db.supply_chain_100gb")

EXCLUDE_COLS = {
    "Late_delivery_risk",
    "_batch_id", "batch_id",
    "Latitude", "Longitude",
    "Customer_Zipcode", "Order_Zipcode",
    "Product_Status", "Department_Id",
    "Category_Id", "Product_Category_Id",
    "Order_Item_Cardprod_Id", "Product_Card_Id",
    "Order_Item_Id", "Order_Id",
    "Customer_Id", "Order_Customer_Id",
}

ALL_NUMERIC_COLS = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, NumericType)
    and f.name not in EXCLUDE_COLS
]

print(f"Numeric columns auto-discovered: {len(ALL_NUMERIC_COLS)}")
for i, c in enumerate(ALL_NUMERIC_COLS):
    print(f"  [{i:02d}] {c}")


Numeric columns auto-discovered: 13
  [00] Days_for_shipping_real
  [01] Days_for_shipment_scheduled
  [02] Benefit_per_order
  [03] Sales_per_customer
  [04] Order_Item_Discount
  [05] Order_Item_Discount_Rate
  [06] Order_Item_Product_Price
  [07] Order_Item_Profit_Ratio
  [08] Order_Item_Quantity
  [09] Sales
  [10] Order_Item_Total
  [11] Order_Profit_Per_Order
  [12] Product_Price


In [0]:

# SECTION 2 — Helper: build feature sample at given fraction

DERIVED_COLS = [
    "delay_gap",
    "discount_to_price_ratio",
    "profit_per_unit",
]
ALL_FEATURES = ALL_NUMERIC_COLS + DERIVED_COLS

def build_feature_sample(fraction, seed=42):
    """
    Sample the table at given fraction, compute derived cols,
    select label + all features, drop nulls, cache.
    Returns cached DataFrame.
    """
    sample_raw = (df
        .sample(fraction=fraction, seed=seed)
        .withColumn("delay_gap",
            (F.col("Days_for_shipping_real") -
             F.col("Days_for_shipment_scheduled"))
             .cast("double"))
        .withColumn("discount_to_price_ratio",
            (F.col("Order_Item_Discount") /
             (F.col("Order_Item_Product_Price") + 0.01))
             .cast("double"))
        .withColumn("profit_per_unit",
            (F.col("Order_Profit_Per_Order") /
             (F.col("Order_Item_Quantity") + 0.01))
             .cast("double"))
    )
    select_exprs = (
        [F.col("Late_delivery_risk").cast("int").alias("label")] +
        [F.col(c).cast("double").alias(c) for c in ALL_FEATURES]
    )
    return sample_raw.select(select_exprs).na.drop().cache()

In [0]:
# PHASE A — Feature discovery on 1GB sample (~0.01 fraction)
# This is where the GA runs — NOT on 100GB.
# Standard practice: discover schema on sample,
# apply to full dataset.


print("\n" + "="*60)
print("PHASE A: Feature Discovery on 1GB Sample")
print("="*60)

PHASE_A_FRACTION = 0.01   # ~1GB
print(f"Sampling {PHASE_A_FRACTION*100:.1f}% of table (~1GB)...")
t_a = time.time()

feature_base = build_feature_sample(PHASE_A_FRACTION)
phase_a_count = feature_base.count()
high_risk_a   = feature_base.filter(F.col("label") == 1).count()

print(f"Phase A sample: {phase_a_count:,} rows")
print(f"High-risk     : {high_risk_a:,} "
      f"({100*high_risk_a/phase_a_count:.1f}%)")
print(f"Sampled in    : {time.time()-t_a:.1f}s")
print("Data in executor memory — GA fitness calls read from RAM.")

#  GA configuration ─
POP_SIZE       = 30
GENERATIONS    = 10
ELITE_FRACTION = 0.3
MUTATION_RATE  = 0.25
MIN_FEATURES   = 2
MAX_FEATURES   = min(12, len(ALL_FEATURES))
n_elite        = int(POP_SIZE * ELITE_FRACTION)

print(f"\nGA config:")
print(f"  Candidates : {len(ALL_FEATURES)}")
print(f"  Population : {POP_SIZE}")
print(f"  Generations: {GENERATIONS}")
print(f"  Elite pool : {n_elite}")

rfc_eval = BinaryClassificationEvaluator(
    labelCol="label", metricName="areaUnderROC")

#  GA operators ─
def fitness(chromosome, data):
    if len(chromosome) < MIN_FEATURES:
        return 0.0
    try:
        asm  = VectorAssembler(
                   inputCols=chromosome,
                   outputCol="features",
                   handleInvalid="skip")
        rf   = RandomForestClassifier(
                   labelCol="label",
                   featuresCol="features",
                   numTrees=50, maxDepth=6, seed=42)
        d         = asm.transform(data)
        tr, ts    = d.randomSplit([0.8, 0.2], seed=42)
        return float(rfc_eval.evaluate(rf.fit(tr).transform(ts)))
    except Exception as e:
        print(f"  Fitness error: {e}")
        return 0.0

def crossover(p1, p2):
    combined = list(set(p1) | set(p2))
    child    = [f for f in combined if random.random() > 0.5]
    if len(child) < MIN_FEATURES:
        child = random.sample(
            combined, min(MIN_FEATURES, len(combined)))
    if len(child) > MAX_FEATURES:
        child = random.sample(child, MAX_FEATURES)
    return child

def mutate(chromosome):
    result     = list(chromosome)
    candidates = [f for f in ALL_FEATURES if f not in result]
    if (random.random() < MUTATION_RATE and candidates
            and len(result) < MAX_FEATURES):
        result.append(random.choice(candidates))
        candidates = [f for f in ALL_FEATURES if f not in result]
    if (random.random() < MUTATION_RATE
            and len(result) > MIN_FEATURES):
        result.remove(random.choice(result))
        candidates = [f for f in ALL_FEATURES if f not in result]
    if (random.random() < MUTATION_RATE and candidates
            and len(result) > MIN_FEATURES):
        result.remove(random.choice(result))
        candidates = [f for f in ALL_FEATURES if f not in result]
        if candidates:
            result.append(random.choice(candidates))
    return result

def inject_random(n):
    return [
        random.sample(ALL_FEATURES,
                      random.randint(MIN_FEATURES, MAX_FEATURES))
        for _ in range(n)
    ]

#  Run GA on Phase A sample 
random.seed(42)
population    = inject_random(POP_SIZE)
best_chr      = []
best_auc      = 0.0
gen_log       = []
all_evaluated = {}

print(f"\nRunning GA on 1GB sample...")
print("="*72)

with mlflow.start_run(run_name="GA_phase_A_1GB"):
    mlflow.set_tag("team_member",    "Mohamed Shiyas")
    mlflow.set_tag("phase",          "A_discovery_1GB")
    mlflow.log_param("sample_fraction", PHASE_A_FRACTION)
    mlflow.log_param("sample_rows",     phase_a_count)
    mlflow.log_param("total_candidates",len(ALL_FEATURES))
    mlflow.log_param("population_size", POP_SIZE)
    mlflow.log_param("generations",     GENERATIONS)

    for gen in range(GENERATIONS):
        gen_start  = time.time()
        cache_hits = 0

        scored = []
        for c in population:
            key = tuple(sorted(c))
            if key in all_evaluated:
                cache_hits += 1
            else:
                all_evaluated[key] = fitness(c, feature_base)
            scored.append((c, all_evaluated[key]))

        scored.sort(key=lambda x: x[1], reverse=True)

        gen_best_chr, gen_best_auc = scored[0]
        gen_avg_auc   = round(
            sum(s for _, s in scored) / len(scored), 4)
        gen_worst_auc = round(scored[-1][1], 4)
        gen_time      = time.time() - gen_start
        unique_chr    = len(set(
            tuple(sorted(c)) for c, _ in scored))
        diversity_pct = round(100 * unique_chr / POP_SIZE, 1)
        new_evals     = POP_SIZE - cache_hits

        gen_log.append({
            "generation":    gen,
            "best_auc":      float(gen_best_auc),
            "avg_auc":       float(gen_avg_auc),
            "worst_auc":     float(gen_worst_auc),
            "features":      str(sorted(gen_best_chr)),
            "n_features":    len(gen_best_chr),
            "diversity_pct": diversity_pct,
            "new_evals":     new_evals,
            "cache_hits":    cache_hits,
            "gen_time_sec":  round(gen_time, 1),
        })

        mlflow.log_metric("best_auc",  gen_best_auc, step=gen)
        mlflow.log_metric("avg_auc",   gen_avg_auc,  step=gen)
        mlflow.log_metric("diversity", diversity_pct, step=gen)

        print(f"Gen {gen:02d} | Best={gen_best_auc:.4f} | "
              f"Avg={gen_avg_auc:.4f} | Div={diversity_pct}% | "
              f"n={len(gen_best_chr)} | {gen_time:.0f}s")
        print(f"  Best: {sorted(gen_best_chr)}")

        if gen_best_auc > best_auc:
            best_auc = gen_best_auc
            best_chr = list(gen_best_chr)

        elite   = [c for c, _ in scored[:n_elite]]
        new_pop = list(elite)
        while len(new_pop) < POP_SIZE - 3:
            p1, p2 = random.sample(elite, 2)
            new_pop.append(mutate(crossover(p1, p2)))
        new_pop.extend(inject_random(3))
        population = new_pop

    mlflow.log_param("best_feature_set", str(sorted(best_chr)))
    mlflow.log_metric("phase_a_best_auc", best_auc)

print(f"\nPhase A complete.")
print(f"Best AUC     : {best_auc:.4f}")
print(f"Best features: {sorted(best_chr)}")

feature_base.unpersist()


PHASE A: Feature Discovery on 1GB Sample
Sampling 1.0% of table (~1GB)...
Phase A sample: 1,019,807 rows
High-risk     : 543,364 (53.3%)
Sampled in    : 31.5s
Data in executor memory — GA fitness calls read from RAM.

GA config:
  Candidates : 16
  Population : 30
  Generations: 10
  Elite pool : 9

Running GA on 1GB sample...
Gen 00 | Best=0.9375 | Avg=0.7900 | Div=100.0% | n=11 | 711s
  Best: ['Benefit_per_order', 'Days_for_shipment_scheduled', 'Days_for_shipping_real', 'Order_Item_Product_Price', 'Order_Item_Profit_Ratio', 'Order_Item_Total', 'Order_Profit_Per_Order', 'Product_Price', 'Sales_per_customer', 'delay_gap', 'discount_to_price_ratio']
Gen 01 | Best=0.9375 | Avg=0.8821 | Div=100.0% | n=11 | 451s
  Best: ['Benefit_per_order', 'Days_for_shipment_scheduled', 'Days_for_shipping_real', 'Order_Item_Product_Price', 'Order_Item_Profit_Ratio', 'Order_Item_Total', 'Order_Profit_Per_Order', 'Product_Price', 'Sales_per_customer', 'delay_gap', 'discount_to_price_ratio']
Gen 02 | Best=

DataFrame[label: int, Days_for_shipping_real: double, Days_for_shipment_scheduled: double, Benefit_per_order: double, Sales_per_customer: double, Order_Item_Discount: double, Order_Item_Discount_Rate: double, Order_Item_Product_Price: double, Order_Item_Profit_Ratio: double, Order_Item_Quantity: double, Sales: double, Order_Item_Total: double, Order_Profit_Per_Order: double, Product_Price: double, delay_gap: double, discount_to_price_ratio: double, profit_per_unit: double]

In [0]:

# PHASE B — Pearson correlation validation
# Independent method confirms GA finding

print("\n" + "="*60)
print("PHASE B: Pearson Correlation Validation")
print("="*60)
print("Independent linear method to validate GA findings.")

# Use fresh small sample for Pandas (driver memory safe)
val_sample_pd = (df
    .sample(fraction=0.001, seed=99)
    .withColumn("delay_gap",
        (F.col("Days_for_shipping_real") -
         F.col("Days_for_shipment_scheduled"))
         .cast("double"))
    .withColumn("discount_to_price_ratio",
        (F.col("Order_Item_Discount") /
         (F.col("Order_Item_Product_Price") + 0.01))
         .cast("double"))
    .withColumn("profit_per_unit",
        (F.col("Order_Profit_Per_Order") /
         (F.col("Order_Item_Quantity") + 0.01))
         .cast("double"))
    .select(
        [F.col("Late_delivery_risk")
          .cast("double").alias("label")] +
        [F.col(c).cast("double").alias(c)
         for c in ALL_FEATURES]
    )
    .na.drop()
    .toPandas()
)

correlations = (val_sample_pd
    .corr()["label"]
    .abs()
    .drop("label")
    .sort_values(ascending=False)
    .reset_index()
)
correlations.columns = ["feature", "abs_pearson_correlation"]

print(f"\nPearson |correlation| with Late_delivery_risk:")
print(correlations.to_string(index=False))

# Agreement check
top_pearson = set(
    correlations.head(len(best_chr))["feature"].tolist())
ga_set      = set(best_chr)
overlap     = top_pearson & ga_set
agreement   = len(overlap) / len(ga_set) * 100

print(f"\nAgreement between GA and Pearson:")
print(f"  GA selected    : {sorted(ga_set)}")
print(f"  Top Pearson    : {sorted(top_pearson)}")
print(f"  Overlap        : {sorted(overlap)}")
print(f"  Agreement rate : {agreement:.0f}%")

if agreement >= 60:
    print("\n  FINDING: GA and Pearson agree — result is robust.")
    print("  The GA discovered features that are both")
    print("  statistically correlated AND predictively powerful,")
    print("  validating the evolutionary search approach.")
else:
    print("\n  FINDING: GA found non-linear features that")
    print("  Pearson missed — demonstrates GA adds value")
    print("  beyond simple correlation analysis.")


PHASE B: Pearson Correlation Validation
Independent linear method to validate GA findings.

Pearson |correlation| with Late_delivery_risk:
                    feature  abs_pearson_correlation
                  delay_gap                 0.715132
     Days_for_shipping_real                 0.416698
Days_for_shipment_scheduled                 0.320281
           Order_Item_Total                 0.007870
                      Sales                 0.007613
   Order_Item_Product_Price                 0.007589
              Product_Price                 0.007539
         Sales_per_customer                 0.007473
        Order_Item_Discount                 0.005000
    Order_Item_Profit_Ratio                 0.002646
     Order_Profit_Per_Order                 0.002176
          Benefit_per_order                 0.001989
    discount_to_price_ratio                 0.001952
            profit_per_unit                 0.001586
   Order_Item_Discount_Rate                 0.001169
        Orde

In [0]:

# PHASE C — AUC stability across data scales
# Proves discovered features generalise from 1GB to 100GB
# This directly addresses "100GB to find 5 features" criticis

print("\n" + "="*60)
print("PHASE C: AUC Stability Across Data Scales")
print("="*60)
print(f"Testing discovered features {sorted(best_chr)}")
print("across multiple data volumes to prove generalisation.")

stability_fractions = {
    "1GB  (0.01)":  0.01,
    "5GB  (0.05)":  0.05,
    "10GB (0.10)":  0.10,
    "25GB (0.25)":  0.25,
    "50GB (0.50)":  0.50,
    "100GB (1.00)": 1.00,
}

stability_rows = []

with mlflow.start_run(run_name="GA_phase_C_stability"):
    mlflow.set_tag("team_member", "Mohamed Shiyas")
    mlflow.set_tag("phase",       "C_stability_validation")
    mlflow.log_param("best_features", str(sorted(best_chr)))

    for label_str, frac in stability_fractions.items():
        vol_gb = round(100 * frac, 1)
        print(f"\n  Testing at {label_str} ({vol_gb}GB)...")

        t_s = time.time()
        scale_sample = build_feature_sample(frac, seed=42)
        scale_count  = scale_sample.count()

        asm_s = VectorAssembler(
                    inputCols=best_chr,
                    outputCol="features",
                    handleInvalid="skip")
        rf_s  = RandomForestClassifier(
                    labelCol="label",
                    featuresCol="features",
                    numTrees=50, maxDepth=6, seed=42)

        data_s    = asm_s.transform(scale_sample)
        tr_s, ts_s = data_s.randomSplit([0.8, 0.2], seed=42)
        model_s   = rf_s.fit(tr_s)
        auc_s     = float(
            rfc_eval.evaluate(model_s.transform(ts_s)))
        elapsed_s = time.time() - t_s

        print(f"    Rows: {scale_count:,} | "
              f"AUC: {auc_s:.4f} | "
              f"Time: {elapsed_s:.0f}s")

        mlflow.log_metric(f"auc_{vol_gb}gb", auc_s)
        mlflow.log_metric(f"time_{vol_gb}gb", elapsed_s)

        stability_rows.append({
            "data_volume":    label_str.strip(),
            "volume_gb":      vol_gb,
            "sample_rows":    scale_count,
            "auc_roc":        round(auc_s, 4),
            "train_time_sec": round(elapsed_s, 1),
        })

        scale_sample.unpersist()

stability_pd = pd.DataFrame(stability_rows)
print("\n AUC STABILITY TABLE ")
print(stability_pd.to_string(index=False))

# Check stability — AUC should not vary by more than 2%
auc_values  = stability_pd["auc_roc"].tolist()
auc_range   = max(auc_values) - min(auc_values)
auc_stable  = auc_range < 0.02

print(f"\nAUC range across scales: {auc_range:.4f}")
if auc_stable:
    print("STABLE: Features discovered on 1GB generalise")
    print("consistently to 100GB — AUC variation < 2%.")
    print("This validates the sampling-first approach.")
else:
    print(f"VARIATION: AUC varies by {auc_range:.4f} across scales.")
    print("Features show some scale sensitivity.")

mlflow.log_metric("auc_stability_range", auc_range)
mlflow.log_param("features_are_stable",  str(auc_stable))

# Save stability results
stability_df = spark.createDataFrame(stability_rows)
spark.sql(
    "DROP TABLE IF EXISTS supply_chain_db.feature_stability")
(stability_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", GOLD_STABILITY_ABFSS)
    .saveAsTable("supply_chain_db.feature_stability"))
print("Stability results saved.")



PHASE C: AUC Stability Across Data Scales
Testing discovered features ['Days_for_shipment_scheduled', 'Days_for_shipping_real', 'Order_Item_Discount', 'Order_Item_Product_Price', 'Order_Item_Profit_Ratio', 'Order_Profit_Per_Order', 'Sales', 'Sales_per_customer', 'delay_gap', 'profit_per_unit']
across multiple data volumes to prove generalisation.

  Testing at 1GB  (0.01) (1.0GB)...
    Rows: 1,019,797 | AUC: 0.9374 | Time: 32s

  Testing at 5GB  (0.05) (5.0GB)...
    Rows: 5,091,721 | AUC: 0.9372 | Time: 42s

  Testing at 10GB (0.10) (10.0GB)...
    Rows: 10,185,006 | AUC: 0.9374 | Time: 70s

  Testing at 25GB (0.25) (25.0GB)...
    Rows: 25,458,778 | AUC: 0.9368 | Time: 148s

  Testing at 50GB (0.50) (50.0GB)...
    Rows: 50,917,514 | AUC: 0.9362 | Time: 289s

  Testing at 100GB (1.00) (100.0GB)...
    Rows: 101,840,453 | AUC: 0.9363 | Time: 609s

 AUC STABILITY TABLE 
 data_volume  volume_gb  sample_rows  auc_roc  train_time_sec
 1GB  (0.01)        1.0      1019797   0.9374        

In [0]:
# ══════════════════════════════════════════════════════════════
# SECTION 3 — Final model + feature importances
# ══════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("Final Model — Feature Importances")
print("="*60)

# Train on 1GB for importances (fast, representative)
final_sample = build_feature_sample(0.01, seed=42)

asm_final = VectorAssembler(
                inputCols=best_chr,
                outputCol="features",
                handleInvalid="skip")
rf_final  = RandomForestClassifier(
                labelCol="label",
                featuresCol="features",
                numTrees=100, maxDepth=6, seed=42)
data_final  = asm_final.transform(final_sample)
model_final = rf_final.fit(data_final)

importances = sorted(
    zip(best_chr, model_final.featureImportances.toArray()),
    key=lambda x: x[1], reverse=True
)

print("\nFeature importances (best chromosome):")
print(f"  {'Feature':<40} {'Importance':>10}  Bar")
print(f"  {'-'*40} {'-'*10}  {'-'*30}")
for feat, imp in importances:
    bar = "█" * int(imp * 40)
    print(f"  {feat:<40} {imp:>10.4f}  {bar}")

final_sample.unpersist()


Final Model — Feature Importances

Feature importances (best chromosome):
  Feature                                  Importance  Bar
  ---------------------------------------- ----------  ------------------------------
  delay_gap                                    0.6729  ██████████████████████████
  Days_for_shipping_real                       0.2179  ████████
  Days_for_shipment_scheduled                  0.1092  ████
  Order_Item_Discount                          0.0000  
  Sales_per_customer                           0.0000  
  Order_Item_Product_Price                     0.0000  
  Order_Item_Profit_Ratio                      0.0000  
  Order_Profit_Per_Order                       0.0000  
  profit_per_unit                              0.0000  
  Sales                                        0.0000  


DataFrame[label: int, Days_for_shipping_real: double, Days_for_shipment_scheduled: double, Benefit_per_order: double, Sales_per_customer: double, Order_Item_Discount: double, Order_Item_Discount_Rate: double, Order_Item_Product_Price: double, Order_Item_Profit_Ratio: double, Order_Item_Quantity: double, Sales: double, Order_Item_Total: double, Order_Profit_Per_Order: double, Product_Price: double, delay_gap: double, discount_to_price_ratio: double, profit_per_unit: double]

In [0]:

# SECTION 4 — Save all output

catalog_rows = [
    {
        "rank":          i + 1,
        "generation":    r["generation"],
        "features":      r["features"],
        "n_features":    r["n_features"],
        "auc_roc":       r["best_auc"],
        "avg_auc":       r["avg_auc"],
        "diversity_pct": r["diversity_pct"],
        "cache_hits":    r["cache_hits"],
        "gen_time_sec":  r["gen_time_sec"],
    }
    for i, r in enumerate(
        sorted(gen_log,
               key=lambda x: x["best_auc"],
               reverse=True)
    )
]
catalog_df = spark.createDataFrame(catalog_rows)
spark.sql("DROP TABLE IF EXISTS supply_chain_db.feature_catalog")
(catalog_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", GOLD_FEATURES_ABFSS)
    .saveAsTable("supply_chain_db.feature_catalog"))
print("\nFeature catalog saved.")

# Feature importances
imp_rows = [
    {"feature": f, "importance": float(imp),
     "rank": i+1, "pearson_rank": int(
         correlations[
             correlations["feature"]==f].index[0])+1
         if f in correlations["feature"].values else -1}
    for i, (f, imp) in enumerate(importances)
]
imp_df = spark.createDataFrame(imp_rows)
spark.sql(
    "DROP TABLE IF EXISTS supply_chain_db.feature_importances")
(imp_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", GOLD_IMP_ABFSS)
    .saveAsTable("supply_chain_db.feature_importances"))
print("Feature importances saved.")

print("\n" + "="*60)
print("NOTEBOOK 05 COMPLETE")
print("="*60)
print(f"  Phase A (GA discovery, 1GB)  : AUC={best_auc:.4f}")
print(f"  Phase B (Pearson validation) : "
      f"Agreement={agreement:.0f}%")
print(f"  Phase C (Scale stability)    : "
      f"Range={auc_range:.4f} "
      f"({'STABLE' if auc_stable else 'VARIABLE'})")
print(f"\n  Best features: {sorted(best_chr)}")
print(f"\n  Thesis contribution:")
print(f"  GA on 1GB sample discovers minimal feature set.")
print(f"  Pearson correlation independently validates result.")
print(f"  AUC stable within {auc_range:.4f} from 1GB to 100GB.")
print(f"  Features proven to generalise at production scale.")
print("\nProceed to Notebook 06.")


Feature catalog saved.
Feature importances saved.

NOTEBOOK 05 COMPLETE
  Phase A (GA discovery, 1GB)  : AUC=0.9376
  Phase B (Pearson validation) : Agreement=80%
  Phase C (Scale stability)    : Range=0.0012 (STABLE)

  Best features: ['Days_for_shipment_scheduled', 'Days_for_shipping_real', 'Order_Item_Discount', 'Order_Item_Product_Price', 'Order_Item_Profit_Ratio', 'Order_Profit_Per_Order', 'Sales', 'Sales_per_customer', 'delay_gap', 'profit_per_unit']

  Thesis contribution:
  GA on 1GB sample discovers minimal feature set.
  Pearson correlation independently validates result.
  AUC stable within 0.0012 from 1GB to 100GB.
  Features proven to generalise at production scale.

Proceed to Notebook 06.
